In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)


+--------+------------------------------+-----------+
|database|tableName                     |isTemporary|
+--------+------------------------------+-----------+
|default |amazon_products               |false      |
|default |amazon_products_day4_delta    |false      |
|default |amazon_products_day4_delta_sql|false      |
|default |day2_top_products             |false      |
|default |day3_amazon_features          |false      |
|default |events_table_day5             |false      |
|default |movies                        |false      |
|default |orders                        |false      |
+--------+------------------------------+-----------+



In [0]:
df = spark.table("workspace.default.amazon_products")
df.show(5, truncate=False)
df.printSchema()

+-------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
bronze_df = df.withColumn("ingestion_ts", F.current_timestamp())


In [0]:
bronze_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_amazon_products")


In [0]:
spark.table("workspace.default.bronze_amazon_products").show(5, truncate=False)

+-------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
bronze = spark.table("workspace.default.bronze_amazon_products")

silver = bronze \
    .withColumn("final_price_clean", F.regexp_replace(F.col("final_price"), "[^0-9.]", "")) \
    .withColumn("initial_price_clean", F.regexp_replace(F.col("initial_price"), "[^0-9.]", "")) \
    .withColumn(
        "final_price_num",
        F.when(F.col("final_price_clean") == "", None)
         .otherwise(F.col("final_price_clean").cast("double"))
    ) \
    .withColumn(
        "initial_price_num",
        F.when(F.col("initial_price_clean") == "", None)
         .otherwise(F.col("initial_price_clean").cast("double"))
    ) \
    .filter(F.col("final_price_num").isNotNull()) \
    .filter(F.col("final_price_num") > 0) \
    .dropDuplicates(["title"]) \
    .withColumn(
        "price_tier",
        F.when(F.col("final_price_num") < 500, "budget")
         .when(F.col("final_price_num") < 2000, "mid")
         .otherwise("premium")
    )



In [0]:
silver.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_amazon_products")


In [0]:
spark.table("workspace.default.silver_amazon_products") \
    .select("title", "final_price", "final_price_num", "price_tier") \
    .show(10, truncate=False)




+------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------------+----------+
|title                                                                                                                                                             |final_price|final_price_num|price_tier|
+------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------------+----------+
|Dixie PerfecTouch 12oz Insulated Paper Hot Cup by GP PRO (Georgia-Pacific), Fits Large Lids, Coffee Haze, 5342CD, 1000 Count (50 Cups Per Pack, 20 Packs Per Case)|"147.70"   |147.7          |budget    |
|Treva 10-Inch Portable Desktop Air Circulation Battery Fan, 2 Speed, Compact Folding & Tilt Design, with AC Adapter (Graphite)                                    |"29.99"    |29.99   

In [0]:
silver_df = spark.table("workspace.default.silver_amazon_products")

group_col = "category" if "category" in silver_df.columns else "brand"

gold_perf = silver_df.groupBy(group_col).agg(
    F.count("*").alias("total_products"),
    F.round(F.avg("final_price_num"), 2).alias("avg_final_price"),
    F.round(F.min("final_price_num"), 2).alias("min_final_price"),
    F.round(F.max("final_price_num"), 2).alias("max_final_price"),
    F.sum(F.when(F.col("price_tier") == "budget", 1).otherwise(0)).alias("budget_products"),
    F.sum(F.when(F.col("price_tier") == "mid", 1).otherwise(0)).alias("mid_products"),
    F.sum(F.when(F.col("price_tier") == "premium", 1).otherwise(0)).alias("premium_products")
).orderBy(F.desc("total_products"))



In [0]:
gold_perf.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_product_performance")


In [0]:
spark.table("workspace.default.gold_product_performance").show(20, truncate=False)


+-----------------+--------------+---------------+---------------+---------------+---------------+------------+----------------+
|brand            |total_products|avg_final_price|min_final_price|max_final_price|budget_products|mid_products|premium_products|
+-----------------+--------------+---------------+---------------+---------------+---------------+------------+----------------+
|Skechers         |10            |52.74          |29.95          |75.99          |10             |0           |0               |
|Amazon Essentials|7             |18.03          |12.8           |26.32          |7              |0           |0               |
|AVERY            |7             |20.32          |10.44          |37.87          |7              |0           |0               |
|New Balance      |6             |64.72          |43.03          |87.99          |6              |0           |0               |
|adidas           |5             |58.9           |32.45          |107.0          |5              

In [0]:
spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)


+--------+------------------------------+-----------+
|database|tableName                     |isTemporary|
+--------+------------------------------+-----------+
|default |amazon_products               |false      |
|default |amazon_products_day4_delta    |false      |
|default |amazon_products_day4_delta_sql|false      |
|default |bronze_amazon_products        |false      |
|default |day2_top_products             |false      |
|default |day3_amazon_features          |false      |
|default |events_table_day5             |false      |
|default |gold_product_performance      |false      |
|default |movies                        |false      |
|default |orders                        |false      |
|default |silver_amazon_products        |false      |
+--------+------------------------------+-----------+

